In [2]:
import pickle
from pathlib import Path
import sys
import os
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
import pandas as pd 
import torch.nn.functional as F


In [3]:

class ECELoss(torch.nn.Module):
    def __init__(self, n_bins=15):
        """
        n_bins: ECE를 계산할 구간(bins)의 개수
        """
        super(ECELoss, self).__init__()
        self.n_bins = n_bins

    def forward(self, prob, labels):
        confidences, predictions = torch.max(prob, 1)
        accuracies = predictions.eq(labels)

        ece = torch.zeros(1, device=prob.device)
        bin_boundaries = torch.linspace(0, 1, self.n_bins + 1)

        for i in range(self.n_bins):
            bin_lower = bin_boundaries[i]
            bin_upper = bin_boundaries[i + 1]
            in_bin = confidences.gt(bin_lower) * confidences.le(bin_upper)
            prop_in_bin = in_bin.float().mean()

            if prop_in_bin.item() > 0:
                accuracy_in_bin = accuracies[in_bin].float().mean()
                avg_confidence_in_bin = confidences[in_bin].mean()
                ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

        return ece


def get_auroc_aupr(in_uncertainty, out_uncertainty):
    # In-Distribution (ID) 데이터는 레이블 0
    in_labels = np.zeros(len(in_uncertainty))
    
    # Out-Of-Distribution (OOD) 데이터는 레이블 1
    out_labels = np.ones(len(out_uncertainty))
    
    # 레이블과 불확실성 값들을 결합
    labels = np.concatenate([in_labels, out_labels])
    uncertainties = np.concatenate([in_uncertainty, out_uncertainty])
    
    # AUROC 계산
    auroc = roc_auc_score(labels, uncertainties)
    
    # Precision-Recall 곡선 계산 및 AUPR 계산
    precision, recall, _ = precision_recall_curve(labels, uncertainties)
    aupr = auc(recall, precision)
    
    return auroc, aupr


In [4]:
import torch.nn as nn

nll_loss = nn.NLLLoss()

def calculate_nll(prob, labels):
    log_prob = F.log_softmax(prob)
    loss = nll_loss(log_prob, labels)
    return loss


In [5]:
current_dir = os.getcwd()
current_dir

'/home/nh/Coding/study'

In [5]:
data = 'cifar10'
method ='0927'
seed = 0
type = 'test'

current_dir = os.getcwd()

new_folder = f"result/{data}/{method}/seed_{seed}"
file = f"cifar10_test_{seed}.pkl"
ood_file = f"cifar10_ood_{seed}.pkl"
full_path = os.path.join(current_dir, new_folder,file)
ood_path = os.path.join(current_dir, new_folder,ood_file)

with open(full_path, 'rb') as f:
    results = pickle.load(f)
with open(ood_path, 'rb') as f:
    ood_results = pickle.load(f)

NameError: name 'os' is not defined

In [19]:
criterion = nn.NLLLoss()

# 손실 계산


probs1 = F.softmax(torch.tensor(results['probs1']))
labels = torch.tensor(results['labels'])


ece_loss = ECELoss(n_bins=10)
ece1 = ece_loss(probs1, labels)

print(f"{method}_{seed}_ECE1: {ece1.item()}")

# NLL 계산
nll1 = calculate_nll(probs1, labels)
print(f"{method}_{seed}_NLL1: {nll1.item()}")


0927_0_ECE1: 0.20923255383968353
0927_0_NLL1: 1.7633570432662964


/tmp/ipykernel_13666/1618203962.py:6: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs1 = F.softmax(torch.tensor(results['probs1']))
/tmp/ipykernel_13666/2803181403.py:6: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  log_prob = F.log_softmax(prob)


In [20]:
criterion = nn.NLLLoss()

# 손실 계산


probs1 = F.softmax(torch.tensor(results['probs1']))
labels = torch.tensor(results['labels'])

probs2 = F.softmax(torch.tensor(results['probs2']))
labels = torch.tensor(results['labels'])

ece_loss = ECELoss(n_bins=10)
ece1 = ece_loss(probs1, labels)
ece2 = ece_loss(probs2, labels)

print(f"{method}_{seed}_ECE1: {ece1.item()}")
print(f"{method}_{seed}_ECE2: {ece2.item()}")

# NLL 계산
nll1 = calculate_nll(probs1, labels)
nll2 = calculate_nll(probs2, labels)
print(f"{method}_{seed}_NLL1: {nll1.item()}")
print(f"{method}_{seed}_NLL2: {nll2.item()}")

0927_0_ECE1: 0.20923255383968353
0927_0_ECE2: 0.281991183757782
0927_0_NLL1: 1.7633570432662964
0927_0_NLL2: 1.7589097023010254


/tmp/ipykernel_13666/3724201800.py:6: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs1 = F.softmax(torch.tensor(results['probs1']))
/tmp/ipykernel_13666/3724201800.py:9: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs2 = F.softmax(torch.tensor(results['probs2']))
/tmp/ipykernel_13666/2803181403.py:6: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  log_prob = F.log_softmax(prob)


In [23]:
probs = torch.tensor(results['probs'])
labels = torch.tensor(results['labels'])
# NLL 계산
nll = calculate_nll(probs, labels)
print(f"{method}_{seed}_NLL: {nll.item()}")

ece_loss = ECELoss(n_bins=10)
ece = ece_loss(probs, labels)
print(f"{method}_{seed}_ECE: {ece.item()}")


KeyError: 'probs'

In [10]:
ood_results.keys()

dict_keys(['probs1', 'probs2', 'preds1', 'preds2', 'uncertainties', 'labels', 'ood_type', 'entropies', 'inference_times'])

In [24]:
df = pd.DataFrame({'unc' :ood_results['uncertainties1'], 'type' : ood_results['ood_type']})
ood_unc1 = df[df.type == 'cifar100_ood']['unc'].values.tolist() 
in_unc1 = results['uncertainties1']


In [25]:
df = pd.DataFrame({'unc' :ood_results['uncertainties2'], 'type' : ood_results['ood_type']})
ood_unc2 = df[df.type == 'cifar100_ood']['unc'].values.tolist() 
in_unc2 = results['uncertainties2']


In [28]:
ood_unc1 = np.array(ood_unc1)
in_unc1 = np.array(in_unc1)
ood_unc2 = np.array(ood_unc2)
in_unc2 = np.array(in_unc2)
# 배열 간 뺄셈 수행



ood_unc = ood_unc2 - ood_unc1
in_unc = in_unc2 - in_unc1

In [113]:

auroc, aupr = get_auroc_aupr(in_unc, ood_unc)
print(f"{method}_AUROC: {auroc:.4f}")
print(f"{method}_AUPR: {aupr:.4f}")

NameError: name 'ood_unc' is not defined

In [21]:
import numpy as np

def compute_ece(probs, labels, n_bins=10):
    # probs: 모델이 예측한 각 클래스의 확률 (shape: [num_samples, num_classes])
    # labels: 실제 클래스 레이블 (shape: [num_samples])
    # n_bins: ECE를 계산하기 위한 빈의 개수
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)  # [0, 0.1, 0.2, ..., 1.0] 형태
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    confidences = np.max(probs, axis=1)  # 각 예측의 최대 확률 (confidence)
    predictions = np.argmax(probs, axis=1)  # 예측된 클래스

    ece = 0.0  # ECE 값을 저장할 변수
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        # 현재 빈에 속한 샘플 찾기
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)  # 해당 빈에 속한 샘플의 비율

        if prop_in_bin > 0:  # 해당 빈에 샘플이 있을 경우에만 계산
            # 빈 내에서의 평균 confidence와 정확도
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            accuracy_in_bin = np.mean(predictions[in_bin] == labels[in_bin])

            # ECE 계산 (빈의 크기에 비례해서 weighted sum)
            ece += prop_in_bin * np.abs(avg_confidence_in_bin - accuracy_in_bin)

    return ece
import torch.nn.functional as F

def compute_nll(probs, labels):
    # probs: 모델이 예측한 각 클래스의 확률 (shape: [num_samples, num_classes])
    # labels: 실제 클래스 레이블 (shape: [num_samples])

    # 실제 정답 레이블에 해당하는 예측 확률만 선택
    num_samples = labels.shape[0]
    predicted_probs = probs[np.arange(num_samples), labels]

    # NLL 계산 (로그 확률의 음수)
    nll = -np.mean(np.log(predicted_probs))
    
    return nll


In [1]:
data = 'cifar10'
method ='my'
seed = 0 #128 #0
type = 'test'

current_dir = os.getcwd()

new_folder = f"result/{data}/{method}/seed_{seed}"
file = f"cifar10_test_{seed}.pkl"
ood_file = f"cifar10_ood_{seed}.pkl"
full_path = os.path.join(current_dir, new_folder,file)
ood_path = os.path.join(current_dir, new_folder,ood_file)

with open(full_path, 'rb') as f:
    results = pickle.load(f)
with open(ood_path, 'rb') as f:
    ood_results = pickle.load(f)

NameError: name 'os' is not defined

In [2]:
print(results.keys())
print(ood_results.keys())

NameError: name 'results' is not defined

In [3]:
results['acc']

NameError: name 'results' is not defined

In [291]:
df = pd.DataFrame({'uncertaintiy' :ood_results['uncertaintiy'], 'type' : ood_results['type']})
in_df = pd.DataFrame({'uncertaintiy' :results['uncertaintiy'], 'type' : results['type']})
ood_unc1 = df[df.type == 'cifar100_ood']['uncertaintiy'].values.tolist() 
in_unc1 = in_df['uncertaintiy'].values.tolist() 
auroc, aupr = get_auroc_aupr(in_unc1, ood_unc1)
print(f"{method}_AUROC: {auroc:.4f}")
print(f"{method}_AUPR: {aupr:.4f}")

ensemble_AUROC: 0.5145
ensemble_AUPR: 0.3778


In [292]:
data = 'cifar10'
method ='vae.'
seed = 3423 #0
type = 'test'

current_dir = os.getcwd()

new_folder = f"result/{data}/{method}/seed_{seed}"
file = f"cifar10_test_{seed}.pkl"
ood_file = f"cifar10_ood_{seed}.pkl"
full_path = os.path.join(current_dir, new_folder,file)
ood_path = os.path.join(current_dir, new_folder,ood_file)

with open(full_path, 'rb') as f:
    results = pickle.load(f)
with open(ood_path, 'rb') as f:
    ood_results = pickle.load(f)

In [273]:

probs1 = torch.tensor(np.array(results['probs']))
labels = torch.tensor(results['labels'])
logit = torch.tensor(results['outputs'])

ece_loss = ECELoss(n_bins=10)
ece1 = ece_loss(probs1, labels)

print(f"{method}_{seed}_ECE1: {ece1.item()}")

# NLL 계산
nll1 = calculate_nll(logit, labels)
print(f"{method}_{seed}_NLL1: {nll1.item()}")

vae._3423_ECE1: 0.2094832807779312
vae._3423_NLL1: 2.288726806640625


/tmp/ipykernel_264173/2803181403.py:6: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  log_prob = F.log_softmax(prob)


In [279]:
print(results.keys())
print(ood_results.keys())

dict_keys(['outputs', 'probs', 'preds', 'labels', 'severity_level', 'distances1', 'distances2', 'inference_times', 'acc'])
dict_keys(['probs', 'preds', 'labels', 'distances1', 'distances2', 'ood_type', 'outputs', 'inference_times'])


In [280]:
results['inference_times']

0.7357935905456543

In [274]:
in_df = pd.DataFrame({'distances' :results['distances1'],'distances2' :results['distances2'], 'type' : results['severity_level']})
in_df

,distances,distances2,type
0,13.411082,0.011568,tensor(0)
1,9.337379,0.016645,tensor(0)
2,23.181105,0.023243,tensor(0)
3,23.187517,0.015135,tensor(0)
4,12.641193,0.014251,tensor(0)
...,...,...,...
8995,14.566754,0.012257,tensor(5)
8996,14.239614,0.011726,tensor(5)
8997,11.928464,0.015645,tensor(5)
8998,9.106527,0.017218,tensor(5)


In [275]:
df = pd.DataFrame({'distances' :results['distances1'], 'type' : results['severity_level']})
df[df.type == 5]['distances'].mean()

11.954851

In [276]:
df = pd.DataFrame({'distances' :ood_results['distances1'], 'type' : ood_results['ood_type']})
df[df.type == 'cifar100_ood']['distances'].mean()

10.1436405

In [277]:
in_df = pd.DataFrame({'distances' :results['distances1'],'distances2' :results['distances2'], 'type' : results['severity_level']})
in_df['u'] = 1  / (in_df['distances'] + 0.0001) *100
in_df

,distances,distances2,type,u
0,13.411082,0.011568,tensor(0),7.456464
1,9.337379,0.016645,tensor(0),10.709529
2,23.181105,0.023243,tensor(0),4.313840
3,23.187517,0.015135,tensor(0),4.312647
4,12.641193,0.014251,tensor(0),7.910583
...,...,...,...,...
8995,14.566754,0.012257,tensor(5),6.864900
8996,14.239614,0.011726,tensor(5),7.022613
8997,11.928464,0.015645,tensor(5),8.383239
8998,9.106527,0.017218,tensor(5),10.981013


In [266]:
df = pd.DataFrame({'distances' :ood_results['distances1'],'distances2' :ood_results['distances2'], 'type' : ood_results['ood_type']})
df['u']= df['distances2'] / (df['distances'] + 0.0001) 
in_df = pd.DataFrame({'distances' :results['distances1'],'distances2' :results['distances2'], 'type' : results['severity_level']})
in_df['u'] = in_df['distances2']  / (in_df['distances'] + 0.0001) 
ood_unc1 = df[df.type == 'cifar100_ood']['u'].values.tolist() 
in_unc1 = in_df['u'].values.tolist() 
auroc, aupr = get_auroc_aupr(in_unc1, ood_unc1)
print(f"{method}_AUROC: {auroc:.4f}")
print(f"{method}_AUPR: {aupr:.4f}")

vae._AUROC: 0.6517
vae._AUPR: 0.4963


In [278]:
df = pd.DataFrame({'distances' :ood_results['distances1'],'distances2' :ood_results['distances2'], 'type' : ood_results['ood_type']})
df['u']= 1 / (df['distances'] + 0.0001) *100
in_df = pd.DataFrame({'distances' :results['distances1'],'distances2' :results['distances2'], 'type' : results['severity_level']})
in_df['u'] = 1 / (in_df['distances'] + 0.0001) *100
ood_unc1 = df[df.type == 'cifar100_ood']['u'].values.tolist() 
in_unc1 = in_df['u'].values.tolist() 
auroc, aupr = get_auroc_aupr(in_unc1, ood_unc1)
print(f"{method}_AUROC: {auroc:.4f}")
print(f"{method}_AUPR: {aupr:.4f}")

vae._AUROC: 0.7234
vae._AUPR: 0.5862


In [268]:
df = pd.DataFrame({'distances' :ood_results['distances1'], 'type' : ood_results['ood_type']},)
ood_unc1 = df[df.type == 'cifar100_ood']['distances'].values.tolist() 
in_unc1 = results['distances1'] 
auroc, aupr = get_auroc_aupr(in_unc1, ood_unc1)
print(f"{method}_AUROC: {auroc:.4f}")
print(f"{method}_AUPR: {aupr:.4f}")

vae._AUROC: 0.2950
vae._AUPR: 0.2865


In [269]:
df = pd.DataFrame({'distances' :results['distances1'], 'type' : results['severity_level'] , 'preds' :results['preds'] , 'labels' :results['labels']})
df['u'] = 1 / (df['distances'] + 0.0001) * 100
df[df.preds != df.labels]['u']

6       12.504838
9        6.991624
14       7.505252
20       5.710257
25      13.879207
          ...    
8979    14.447922
8981    15.997065
8982    11.379002
8989     8.918637
8994     8.432462
Name: u, Length: 2585, dtype: float32

In [271]:
df_sorted = df.sort_values(by='u', ascending = False)
df_sorted[:30]

,distances,type,preds,labels,u
1831,120.633629,tensor(1),8,8,0.828956
5514,115.308090,tensor(3),8,8,0.867241
1129,113.276917,tensor(0),8,8,0.882792
29,109.964500,tensor(0),8,8,0.909384
1849,104.421753,tensor(1),8,8,0.957654
2761,104.258469,tensor(1),8,8,0.959154
1237,101.599701,tensor(0),8,8,0.984254
6303,96.206764,tensor(4),8,8,1.039427
2556,93.667694,tensor(1),8,8,1.067603
2721,92.538155,tensor(1),8,8,1.080634


In [ ]:
df = pd.DataFrame({'distances' :ood_results['distances1'],'distances2' :ood_results['distances2'], 'type' : ood_results['ood_type']})
df['u']= df['distances2'] / (df['distances'] + 0.0001) 
in_df = pd.DataFrame({'distances' :results['distances1'],'distances2' :results['distances2'], 'type' : results['severity_level']})
in_df['u'] = in_df['distances2']  / (in_df['distances'] + 0.0001) 
ood_unc1 = df[df.type == 'cifar100_ood']['u'].values.tolist() 
in_unc1 = in_df['u'].values.tolist() 
auroc, aupr = get_auroc_aupr(in_unc1, ood_unc1)
print(f"{method}_AUROC: {auroc:.4f}")
print(f"{method}_AUPR: {aupr:.4f}")

vae_AUROC: 0.3522
vae_AUPR: 0.3057
